<a href="https://colab.research.google.com/github/ZoeyZhu2/data_dashboard/blob/main/data_dashboard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Goals:
Display buildout progress per bid in RDOF.

Current progress: MD, VA, DE

Desciption:
On a national-scale map (more than 1 state),
All census block groups depicted as hexagons.
Display baseline data (June 2022 from NBM) of census block groups served colored dark.
Display planned census block groups served according to RDOF bids colored in a light color that corresponds to the particular RDOF bid.
Display newly served (cumulative) census block groups up to a particular time (July 2022 to December 2025) colored in a dark color.
Color each census block group with saturation corresponding to percentage served.
Every bid should get the same color.


State-scale map:
All RDOF locations depicted as hexagons.
Display baseline data (June 2022 from NBM) of RDOF locations served colored dark.
Display planned RDOF locations served according to RDOF bids colored in a light color that corresponds to the particular RDOF bid (each bid gets its own color)
Display newly served (cumulative) RDOF locations up to a particular time (July 2022 to December 2025) colored in a dark color that corresponds to that particular RDOF bid.


In 2 table below the map,
By CBG or RDOF location:
Show the bid awardee (company),
number of planned CBGs/location,
number of currently built out CBGs/locations (cumulative) that corresponds to the time chosen
number of baseline CBGs/locations, and total number of CBGs/locations.


Above the map, toggle options for which state to show data (select 1-50), toggle options for which companies/bids to show data (1 to all), and a slider from June 2022 to December 2025 in 1 month increments to select the "current time" to show cumulative progress. Also pick view: defaults to CBG for more than one state and RDOF location for 1 state.

In [ ]:
# REQUIRED: authenticate with GCP project
GCP_PROJECT_ID = "broadband-data"  #@param {type:"string"}

from google.colab import auth
from google.colab import userdata

auth.authenticate_user(project_id=GCP_PROJECT_ID)

In [ ]:
# Authenticate GCP Client
from google.cloud import bigquery

client = bigquery.Client(project=GCP_PROJECT_ID)
print("Authenticated & Output Directory Initialized!")

In [ ]:
bdc_query = """
WITH rdof_blocks AS (
  SELECT DISTINCT census_block, applicant, LTRIM(frn,'0') AS rdof_frn, sac, county
  FROM `broadband-data.rdof.applicants_hist_2023_cb`
  WHERE state_abbr = 'MD'
),
bids AS (
  SELECT sac, LTRIM(frn,'0') AS rdof_frn,
         SUM(bid_amount) AS bid_amount,
         SUM(n_locations) AS bid_locations,
         STRING_AGG(DISTINCT da_number) AS da_number
  FROM `broadband-data.rdof.applicants_hist_2023`
  WHERE state_abbr = 'MD'
  GROUP BY sac, rdof_frn
),
defaulted AS (
  SELECT DISTINCT census_block
  FROM `broadband-data.rdof.defaults_dedup_cb`
  WHERE state_abbr = 'MD'
),
rdof_locs AS (
  SELECT f.location_id, f.block_geoid, f.county_geoid, f.latitude, f.longitude,
         b.applicant, b.rdof_frn, b.sac,
         bd.bid_amount, bd.bid_locations, bd.da_number,
         d.census_block IS NOT NULL AS block_defaulted
  FROM `broadband-data.bsl_fabric.v7` f
  JOIN rdof_blocks b ON f.block_geoid = b.census_block
  LEFT JOIN bids bd ON bd.sac = b.sac AND bd.rdof_frn = b.rdof_frn
  LEFT JOIN defaulted d ON d.census_block = b.census_block
  WHERE f.bsl_flag AND f.state = 'MD'
)
SELECT
  a.period, a.release,
  l.location_id, l.block_geoid, l.county_geoid, l.latitude, l.longitude,
  l.applicant AS rdof_awardee, l.rdof_frn, l.sac, l.bid_amount, l.bid_locations,
  l.da_number, l.block_defaulted,
  a.frn AS provider_frn, a.provider_id, a.brand_name, a.technology,
  a.max_advertised_download_speed AS down, a.max_advertised_upload_speed AS up,
  a.low_latency, a.business_residential_code,
  LTRIM(a.frn,'0') = l.rdof_frn AS is_awardee
FROM rdof_locs l
LEFT JOIN `broadband-data.fcc_bdc.nbm_hive` a
  ON a.location_id = l.location_id AND a.state_usps = 'MD'
"""

df_bdc = client.query(bdc_query).to_dataframe()
df_bdc.to_parquet("md_rdof_bdc_long.parquet")   # saves in Colab session storage
print(df_bdc.shape, df_bdc.period.nunique(), "periods")

In [ ]:
"""
Shows how many of each awardee’s RDOF locations it reported serving at 100 Mbps down and 20 Mbps up, in each period.
Shows percentage of the awardee’s total locations.
"""

d = df_bdc.copy()
d["awardee_100_20"] = (d.is_awardee == True) & (d.down >= 100) & (d.up >= 20)
total = d.groupby("rdof_awardee").location_id.nunique().rename("rdof_locations")
prog = (d[d.awardee_100_20].groupby(["rdof_awardee", "period"]).location_id.nunique()
        .unstack("period").fillna(0).join(total))
for c in prog.columns[:-1]:
    prog[c + "_pct"] = (100 * prog[c] / prog["rdof_locations"]).round(1)
prog

In [ ]:
# ---- Cell 1: one row per RDOF location, ALL states ----
import os, pandas as pd
from google.cloud import bigquery

BASELINE = "2022_06"       # baseline = served by the awardee in this NBM filing
SERVED_THRESH = 0.80       # share of locations for a hexagon / block group to count as "served"
STATES = None              # None = every state with RDOF awards; or e.g. ["MD", "VA", "DE"] to test
MAX_GB = 800               # safety cap: BigQuery refuses to run above this
CACHE = ("rdof_locations_all.parquet" if STATES is None
         else "rdof_locations_" + "_".join(STATES) + ".parquet")   # delete the file to re-query

SQL = """
WITH rdof_blocks AS (
  SELECT DISTINCT census_block, applicant, LTRIM(frn, '0') AS rdof_frn, sac
  FROM `broadband-data.rdof.applicants_hist_2023_cb`
  WHERE state_abbr IN UNNEST(@states)
    AND census_block NOT IN (SELECT census_block FROM `broadband-data.rdof.defaults_dedup_cb`)
),
rdof_locs AS (
  SELECT f.location_id, f.state, DIV(f.block_geoid, 1000) AS cbg, f.latitude, f.longitude, f.h3_9,
         b.applicant AS awardee, b.rdof_frn, b.sac
  FROM `broadband-data.bsl_fabric.v7` f
  JOIN rdof_blocks b ON f.block_geoid = b.census_block
  WHERE f.bsl_flag AND f.state IN UNNEST(@states)
),
first_served AS (
  SELECT l.location_id, MIN(a.period) AS first_served_period
  FROM rdof_locs l
  JOIN `broadband-data.fcc_bdc.nbm_hive` a
    ON a.location_id = l.location_id
   AND a.state_usps = l.state
   AND LTRIM(a.frn, '0') = l.rdof_frn
  WHERE a.state_usps IN UNNEST(@states)
   -- optional speed filter: AND a.max_advertised_download_speed >= 100
   --                        AND a.max_advertised_upload_speed >= 20
  GROUP BY l.location_id
)
SELECT l.*, fs.first_served_period
FROM rdof_locs l
LEFT JOIN first_served fs USING (location_id)
"""

all_states = client.query("""SELECT DISTINCT state_abbr
    FROM `broadband-data.rdof.applicants_hist_2023_cb` ORDER BY 1""").to_dataframe().state_abbr.tolist()
states_to_load = STATES or all_states
params = [bigquery.ArrayQueryParameter("states", "STRING", states_to_load)]
print(len(states_to_load), "states:", ", ".join(states_to_load))

if os.path.exists(CACHE):
    loc = pd.read_parquet(CACHE)
    print("Loaded from cache:", CACHE)
else:
    dry = client.query(SQL, job_config=bigquery.QueryJobConfig(
        query_parameters=params, dry_run=True, use_query_cache=False))
    print(f"Estimated scan: {dry.total_bytes_processed / 1e9:,.0f} GB "
          "(may show 0 for external tables, so the cap below is the real safety net)")
    loc = client.query(SQL, job_config=bigquery.QueryJobConfig(
        query_parameters=params, maximum_bytes_billed=int(MAX_GB * 1e9))).to_dataframe()
    loc["latitude"] = loc.latitude.astype("float32")
    loc["longitude"] = loc.longitude.astype("float32")
    loc.to_parquet(CACHE)
    print("Saved cache:", CACHE)

periods = sorted(client.query(f"""
    SELECT DISTINCT period FROM `broadband-data.fcc_bdc.nbm_hive`
    WHERE state_usps = '{states_to_load[0]}'""").to_dataframe().period)

summary = loc.groupby("state").agg(
    locations=("location_id", "size"), block_groups=("cbg", "nunique"),
    awardees=("awardee", "nunique"), ever_served=("first_served_period", "count"))
print(f"\n{len(loc):,} locations | {loc.awardee.nunique()} awardees | periods {periods[0]} to {periods[-1]}")
summary

In [ ]:
# ---- Cell 2: block group table, hexagons, status rules, colors ----
!pip -q install "h3>=4"
import h3, colorsys, numpy as np, pandas as pd

CBG_RES = 6            # block group hexagons: 5 = big, 6 = medium, 7 = small

PIDX = {p: i for i, p in enumerate(periods)}      # period -> 0, 1, 2 ...
BASE_I = PIDX[BASELINE]

if "first_served_period" in loc.columns:          # compact storage; safe to re-run
    loc["fs_i"] = loc.first_served_period.map(PIDX).fillna(99).astype("int8")   # 99 = never served
    loc = loc.drop(columns=["first_served_period"])
for c in ("state", "awardee"):
    loc[c] = loc[c].astype("category")

# --- block groups: location counts, cumulative served per period (computed once) ---
cnt = (loc.groupby(["state", "awardee", "sac", "cbg", "fs_i"], observed=True)
          .size().unstack("fs_i", fill_value=0))
cbg_tab = (loc.groupby(["state", "awardee", "sac", "cbg"], observed=True)
              .agg(n_total=("location_id", "size"), lat=("latitude", "mean"), lon=("longitude", "mean")))
cum_np = (cnt.reindex(columns=range(len(periods)), fill_value=0).cumsum(axis=1)
             .reindex(cbg_tab.index).to_numpy())          # served by period p = first served <= p
cbg_tab["n_base"] = cum_np[:, BASE_I]
bid_states = cbg_tab.reset_index()[["state", "awardee"]].drop_duplicates()   # which bids are in which state

# --- one distinct hexagon per block group, placed once ---
def place_hexes(info):
    occupied, cells = set(), {}
    for idx, r in info.sort_values("n_total", ascending=False).iterrows():
        home = h3.latlng_to_cell(r.lat, r.lon, CBG_RES)
        cell, k = home, 0
        while cell in occupied:
            k += 1
            free = [c for c in h3.grid_ring(home, k) if c not in occupied]
            if free:
                cell = min(free, key=lambda c: (h3.cell_to_latlng(c)[0] - r.lat) ** 2
                                             + (h3.cell_to_latlng(c)[1] - r.lon) ** 2)
        occupied.add(cell)
        cells[idx] = cell
    return cells

CBG_HEX = place_hexes(cbg_tab)             # key: (state, awardee, sac, cbg) -> hexagon id

_geom = {}
def geom_of(h):
    if h not in _geom:
        ring = [[round(lng, 5), round(lat, 5)] for lat, lng in h3.cell_to_boundary(h)]  # GeoJSON: [lng, lat]
        ring.append(ring[0])
        _geom[h] = {"type": "Polygon", "coordinates": [ring]}
    return _geom[h]

# --- colors ---
# One hue per bid, in a fixed order, reassigned per map from the bids on screen.
# TODO: reads well for ~8 bids on screen at once; hues repeat beyond that.
BASE_HEX = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
def hue_map(bid_list):
    return {b: colorsys.rgb_to_hls(*[int(BASE_HEX[i % 8][j:j + 2], 16) / 255 for j in (1, 3, 5)])[0]
            for i, b in enumerate(bid_list)}
HUE = {}
LIGHTNESS = {"planned": 0.82, "new": 0.58, "baseline": 0.34}   # light / medium / darkest
LABEL = {"planned": "Planned", "new": "Newly served (cumulative)",
         "baseline": f"Baseline (served by {BASELINE})"}
NATIONAL_HUE = colorsys.rgb_to_hls(*[int("#2a78d6"[j:j + 2], 16) / 255 for j in (1, 3, 5)])[0]

def shade(awardee, cls, pct, single_color=False):
    """Hue = bid (or one shared blue if single_color), lightness = status,
    saturation = share of locations served (20%-100%)."""
    hue = NATIONAL_HUE if single_color else HUE[awardee]
    r, g, b = colorsys.hls_to_rgb(hue, LIGHTNESS[cls], 0.20 + 0.80 * float(pct))
    return "#%02x%02x%02x" % (int(r * 255), int(g * 255), int(b * 255))

# --- status rules ---
def classify(t):
    t["pct"] = t.n_served / t.n_total
    t["pct_base"] = t.n_base / t.n_total
    t["cls"] = np.where(t.pct_base >= SERVED_THRESH, "baseline",
                np.where(t.pct >= SERVED_THRESH, "new", "planned"))
    return t

def block_group_table(states, companies, p):
    pi = PIDX[p]
    idx = cbg_tab.index
    m = idx.get_level_values("state").isin(states) & idx.get_level_values("awardee").isin(companies)
    t = cbg_tab[m].reset_index()
    t["n_served"] = cum_np[m, pi]
    t["hex"] = [CBG_HEX[k] for k in zip(t.state, t.awardee, t.sac, t.cbg)]
    t["area"] = t.cbg.astype(str).str.zfill(12)
    return classify(t)

# --- location hexagons at any size, computed fast from integer H3 ids ---
H3_INT = np.fromiter((int(c, 16) for c in loc.h3_9), dtype=np.uint64, count=len(loc))

def parent_int(cells, res):
    """H3 parent at `res` for an array of resolution-9 cell ids (pure bit arithmetic)."""
    keep = ~np.uint64(0xF << 52)
    return (cells & keep) | np.uint64(res << 52) | np.uint64((1 << (3 * (15 - res))) - 1)

def location_table(idx, p, res):
    """Hexagons of size `res` for the locations at row positions `idx` (the map viewport)."""
    sub = loc.iloc[idx]
    pi = PIDX[p]
    d = sub.assign(cell=parent_int(H3_INT[idx], res), sp=sub.fs_i <= pi, sb=sub.fs_i <= BASE_I)
    t = (d.groupby(["awardee", "sac", "cell"], observed=True)
          .agg(n_total=("location_id", "size"), n_served=("sp", "sum"), n_base=("sb", "sum"),
               lat=("latitude", "mean"), lon=("longitude", "mean")).reset_index())
    t["hex"] = [format(int(c), "x") for c in t.cell]
    t["area"] = t.hex
    return classify(t)

def filing_for(month):
    """Slider month 'YYYY-MM' -> latest NBM filing at or before it ('YYYY_MM')."""
    return [p for p in periods if p.replace("_", "-") <= month][-1]

In [ ]:
# ---- Cell 3: interactive map (zoom in for locations) + tables + toggles ----
!pip -q install ipyleaflet
import ipywidgets as widgets, numpy as np, pandas as pd
from IPython.display import display, clear_output
from ipyleaflet import Map, GeoJSON, TileLayer, WidgetControl
try:
    from google.colab import output
    output.enable_custom_widget_manager()      # lets Colab show ipyleaflet
except Exception:
    pass

MAX_VIEW_HEXES = 15000      # more hexagons than this in view -> keep block groups
MAX_VIEW_LOCS = 2_000_000   # more locations than this in view -> keep block groups
MAX_BID_COLORS = 8          # per-bid colors only when the view has this many bids or fewer

def loc_res_for(z):
    """Hexagon size by zoom: coarse when zoomed out, address-sized when zoomed in."""
    z = int(z)
    if z >= 11:
        return 9
    return {10: 8, 9: 7, 8: 6, 7: 5}.get(z)      # None below zoom 7 -> block groups

# If the background tiles are blocked in your notebook, remove the TileLayer below.
TILES = "https://server.arcgisonline.com/ArcGIS/rest/services/Canvas/World_Light_Gray_Base/MapServer/tile/{z}/{y}/{x}"
ATTR = "Tiles © Esri, HERE, Garmin, © OpenStreetMap contributors"
COLS = ["planned", "new", "baseline"]
LAT, LON = loc.latitude.to_numpy(), loc.longitude.to_numpy()

months = [str(x) for x in pd.period_range(BASELINE.replace("_", "-"),
                                          periods[-1].replace("_", "-"), freq="M")]
states = sorted(bid_states.state.astype(str).unique())
companies = sorted(bid_states.awardee.astype(str).unique())

def _bids_in(sel):
    return sorted(bid_states[bid_states.state.astype(str).isin(sel)].awardee.astype(str).unique())

# ---------------- the map ----------------
m = Map(center=(39.0, -90.0), zoom=4, scroll_wheel_zoom=True,
        layout=widgets.Layout(height="650px"))
m.layers = (TileLayer(url=TILES, attribution=ATTR),)

info = widgets.HTML("<div style='font:12px sans-serif;background:#fff;padding:6px 8px;"
                    "border:1px solid #ccc;border-radius:4px'>Hover over a hexagon</div>")
legend = widgets.HTML("")
legend.layout.display = "none"
m.add_control(WidgetControl(widget=info, position="topright"))
m.add_control(WidgetControl(widget=legend, position="bottomleft"))

def on_hover(feature=None, **kwargs):
    if not feature:
        return
    p = feature["properties"]
    info.value = ("<div style='font:12px sans-serif;background:#fff;padding:6px 8px;"
                  "border:1px solid #ccc;border-radius:4px'>"
                  f"<b>{p['bid']}</b><br>{p['status']}<br>{p['pct_served']}% served "
                  f"({p['n_served']:,} of {p['n_total']:,} locations)<br>"
                  f"<span style='color:#777'>{p['area']}</span></div>")

def make_layer(t):
    feats = [{"type": "Feature", "geometry": geom_of(r.hex), "properties": {
                "bid": r.awardee, "status": LABEL[r.cls], "pct_served": round(100 * float(r.pct), 1),
                "n_served": int(r.n_served), "n_total": int(r.n_total),
                "area": r.area, "color": r.color}} for r in t.itertuples()]
    layer = GeoJSON(
        data={"type": "FeatureCollection", "features": feats},
        hover_style={"weight": 2, "color": "#222222"},
        style_callback=lambda f: {"fillColor": f["properties"]["color"], "fillOpacity": 0.9,
                                  "color": "#ffffff", "weight": 0.4})
    layer.on_hover(on_hover)
    return layer

_current = {"layer": None}
def show_layer(layer):
    if _current["layer"] is layer:
        return
    if _current["layer"] is not None:
        m.remove_layer(_current["layer"])
    m.add_layer(layer)
    _current["layer"] = layer

S = {"p": None, "cb": None, "bg_layer": None, "sel_mask": None}
note = widgets.HTML("")
banner = widgets.HTML("")
table_out = widgets.Output()

def swatches(awardee):
    return " ".join(
        f'<span style="color:{shade(awardee, c, 1.0, False)}">&#9632;</span>{LABEL[c].split(" (")[0]}'
        for c in ("baseline", "new", "planned"))

def update_detail(change=None):
    """Block groups when zoomed far out; otherwise location hexagons sized to the zoom level."""
    if S["bg_layer"] is None:
        return
    legend.layout.display = "none"
    res = (loc_res_for(m.zoom) if view_w.value == "Auto"
           and m.bounds is not None and len(m.bounds) == 2 else None)
    if res is None:
        show_layer(S["bg_layer"])
        note.value = (f"<i>Showing block groups. Zoom in to level 7 or closer (now {int(m.zoom)}) "
                      "to see locations.</i>" if view_w.value == "Auto"
                      else "<i>Showing block groups.</i>")
        return
    (s, w), (n, e) = m.bounds
    idx = np.flatnonzero(S["sel_mask"] & (LAT >= s) & (LAT <= n) & (LON >= w) & (LON <= e))
    if len(idx) == 0:
        show_layer(S["bg_layer"]); note.value = "<i>No RDOF locations in view.</i>"; return
    if len(idx) > MAX_VIEW_LOCS:
        show_layer(S["bg_layer"])
        note.value = f"<i>{len(idx):,} locations in view. Zoom in to see them; showing block groups.</i>"
        return
    t = location_table(idx, S["p"], res)
    if len(t) > MAX_VIEW_HEXES:
        show_layer(S["bg_layer"])
        note.value = (f"<i>{len(t):,} hexagons in view. Zoom in further to see locations; "
                      "showing block groups.</i>")
        return
    bids = sorted(t.awardee.astype(str).unique())
    per_bid = len(bids) <= MAX_BID_COLORS
    if per_bid:
        HUE.clear(); HUE.update(hue_map(bids))
    t["color"] = [shade(r.awardee, r.cls, r.pct, not per_bid) for r in t.itertuples()]
    show_layer(make_layer(t))
    what = "individual locations" if res == 9 else "groups of locations"
    note.value = (f"<i>Showing {len(t):,} hexagons of {what} in view (zoom {int(m.zoom)}). "
                  "Zoom in further for single addresses.</i>" if res < 9
                  else f"<i>Showing {len(t):,} location hexagons in view (zoom {int(m.zoom)}).</i>")
    if per_bid:
        legend.value = ("<div style='font:12px sans-serif;background:#fff;padding:8px 10px;"
                        "border:1px solid #ccc;border-radius:6px;max-height:40vh;overflow-y:auto'>"
                        + "".join(f"<div><b>{b}</b> {swatches(b)}</div>" for b in bids)
                        + "<div><i>Saturation = % served</i></div></div>")
        legend.layout.display = "block"

m.observe(update_detail, names=["bounds"])

# ---------------- tables and banner ----------------
def banner_html(sel, n_shown, n_avail):
    chip = ('<span style="display:inline-block;margin:0 4px 4px 0;padding:2px 10px;border-radius:12px;'
            'font:600 13px sans-serif;{}">{}</span>')
    on  = "".join(chip.format("background:#1f7a3a;color:#fff", s) for s in sel) or "<i>none</i>"
    off = "".join(chip.format("background:#e6e6e6;color:#777;text-decoration:line-through", s)
                  for s in states if s not in sel) or "<i>none</i>"
    return (f'<div style="font:13px sans-serif;margin:6px 0"><b>Showing:</b> {on} '
            f'&nbsp;&nbsp; <b>Not showing:</b> {off} &nbsp;&nbsp; '
            f'<b>Companies:</b> {n_shown} of {n_avail}</div>')

def show_tables(cb, group):
    cb = cb.copy()
    cb["planned_n"] = cb.n_total - cb.n_served
    cb["new_n"] = cb.n_served - cb.n_base
    bg = (cb.pivot_table(index=group, columns="cls", values="cbg", aggfunc="count",
                         fill_value=0, observed=True).reindex(columns=COLS, fill_value=0))
    lc = cb.groupby(group, observed=True)[["planned_n", "new_n", "n_base"]].sum()
    for tab in (bg, lc):
        tab.columns = ["Planned", "Built since baseline (cumulative)", "Baseline"]
        tab["Total"] = tab.sum(axis=1)
        if group == "state":
            tab.loc["All states"] = tab.sum()
    with table_out:
        clear_output(wait=True)
        print("Block groups"); display(bg)
        print("Locations");    display(lc)

# ---------------- refresh when a control changes ----------------
def refresh(fit=False):
    sel = [s for s in states_text.value.split(",") if s]
    comp = list(company_w.value)
    avail = _bids_in(sel)
    banner.value = banner_html(sel, len([c for c in comp if c in set(avail)]), len(avail))
    S["bg_layer"] = None
    if not sel or not comp:
        if _current["layer"] is not None:
            m.remove_layer(_current["layer"]); _current["layer"] = None
        note.value = "<i>Select at least one state and one company.</i>"
        with table_out:
            clear_output()
        return
    p = filing_for(month_w.value)
    cb = block_group_table(sel, comp, p)
    if cb.empty:
        note.value = "<i>No RDOF block groups for this selection.</i>"; return
    cb["color"] = [shade(r.awardee, r.cls, r.pct, True) for r in cb.itertuples()]   # one shared color
    S.update(p=p, cb=cb, bg_layer=make_layer(cb),
             sel_mask=(loc.state.isin(sel) & loc.awardee.isin(comp)).to_numpy())
    show_tables(cb, "state" if len(sel) > 1 else "awardee")
    if fit:
        try:
            m.fit_bounds([[float(cb.lat.min()), float(cb.lon.min())],
                          [float(cb.lat.max()), float(cb.lon.max())]])
        except Exception:
            m.center = (float(cb.lat.mean()), float(cb.lon.mean())); m.zoom = 6
    update_detail()

# ---------------- controls ----------------
state_boxes = {s: widgets.ToggleButton(value=True, description=s, icon="check",
                                       button_style="success", tooltip=f"Show/hide {s}",
                                       layout=widgets.Layout(width="70px"))
               for s in states}
all_btn  = widgets.Button(description="All",  layout=widgets.Layout(width="70px"))
none_btn = widgets.Button(description="None", layout=widgets.Layout(width="70px"))
states_text = widgets.Text(value=",".join(states), layout=widgets.Layout(display="none"))  # hidden carrier
company_w = widgets.SelectMultiple(options=companies, value=tuple(companies),
                                   description="Companies", rows=min(max(len(companies), 2), 8),
                                   layout=widgets.Layout(width="420px"))
view_w = widgets.Dropdown(options=["Auto", "Block groups only"], value="Auto",
                          description="View", layout=widgets.Layout(width="320px"))
month_w = widgets.SelectionSlider(options=months, value=months[-1], description="As of",
                                  layout=widgets.Layout(width="520px"))

_bulk = False
_syncing = False
def _current_states():
    return ",".join(s for s, b in state_boxes.items() if b.value)

def on_toggle(change):
    b = change["owner"]
    b.button_style = "success" if b.value else ""
    b.icon = "check" if b.value else ""
    if not _bulk:
        states_text.value = _current_states()

def set_all(value):
    global _bulk
    _bulk = True
    for b in state_boxes.values():
        b.value = value
    _bulk = False
    states_text.value = _current_states()

def sync_companies(change=None):
    """Companies list = bids in the selected state(s); select all of them, then redraw."""
    global _syncing
    sel = [s for s in states_text.value.split(",") if s]
    opts = _bids_in(sel)
    _syncing = True
    company_w.value = ()
    company_w.options = opts
    company_w.rows = min(max(len(opts), 2), 8)
    company_w.value = tuple(opts)
    _syncing = False
    refresh(fit=True)

for b in state_boxes.values():
    b.observe(on_toggle, "value")
all_btn.on_click(lambda _: set_all(True))
none_btn.on_click(lambda _: set_all(False))
states_text.observe(sync_companies, "value")
company_w.observe(lambda c: None if _syncing else refresh(fit=True), "value")
month_w.observe(lambda c: refresh(fit=False), "value")
view_w.observe(update_detail, "value")

display(widgets.VBox([
    widgets.HBox([widgets.Label("States:"), all_btn, none_btn]),
    widgets.HBox(list(state_boxes.values()), layout=widgets.Layout(flex_flow="row wrap")),
    company_w, view_w, month_w, banner, m, note, table_out, states_text]))
sync_companies()